# Nuevos Qubits: Fluxonium, Spin-Si y Majorana

**Laboratorio 36 · Nivel muy avanzado**

La segunda generación de hardware cuántico supera las limitaciones del transmon
con arquitecturas más anarmónicas y coherentes. Este laboratorio compara:

- **Fluxonium**: alta anharmonicidad, T₂ > 1 ms
- **Spin-Si (quantum dot)**: compatibilidad CMOS, densidad extrema
- **Majorana (topológico)**: error intrínsecamente protegido
- **Proyecciones 2025-2030**: roadmaps de IBM, Google, Microsoft, Intel


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

print('Entorno cargado')


## 1. Hamiltoniano Fluxonium vs Transmon

El **transmon** reduce la carga-sensitividad de la Cooper-pair box añadiendo un condensador shunt.
El **fluxonium** va más lejos: añade una inductancia superinductora que permite controlar
el punto de operación con flujo externo $\Phi_{\text{ext}}$.

$$H_{\text{fluxonium}} = 4E_C(\hat{n} - n_g)^2 - E_J\cos(\hat{\varphi} - 2\pi\Phi_{\text{ext}}/\Phi_0) + \frac{1}{2}E_L\hat{\varphi}^2$$


In [ ]:
def transmon_eigenvalues(EJ_EC_ratio: float, n_levels: int = 5) -> np.ndarray:
    """
    Calcula autovalores del transmon truncando en n_levels estados de carga.
    EJ/EC: razón de energía Josephson a carga (típico: 20-80).
    """
    N = 30  # número de estados de carga incluidos (-N, ..., N)
    n_arr = np.arange(-N, N+1)
    # Hamiltoniano de carga diagonal
    H_C = np.diag(4.0 * n_arr**2)  # EC = 1 (normalizado)
    # Acoplamiento Josephson (nearest neighbor en espacio de carga)
    H_J = np.zeros((2*N+1, 2*N+1))
    for i in range(2*N):
        H_J[i, i+1] = H_J[i+1, i] = -EJ_EC_ratio / 2
    H = H_C + H_J
    eigvals = np.sort(np.linalg.eigvalsh(H))
    return eigvals[:n_levels] - eigvals[0]

def anharmonicidad(EJ_EC: float) -> float:
    """α = (E_21 - E_10) / E_10 — debe ser < 0 para transmon."""
    E = transmon_eigenvalues(EJ_EC, 4)
    E10 = E[1] - E[0]
    E21 = E[2] - E[1]
    return (E21 - E10) / E10

# Comparativa: anharmonicidad vs EJ/EC
EJ_EC_vals = np.linspace(1, 100, 100)
anharmon_vals = [anharmonicidad(r) for r in EJ_EC_vals]

# Fluxonium: anharmonicidad típica ≈ -20% a -80% (mucho mayor)
fluxonium_anh = np.array([-0.4, -0.5, -0.6, -0.7, -0.5, -0.4])  # en punto óptimo
fluxonium_EJ_EC = np.array([0.5, 1.0, 2.0, 3.0, 5.0, 10.0])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(EJ_EC_vals, np.abs(anharmon_vals)*100, 'b-', lw=2, label='Transmon')
ax.axhline(5, color='gray', ls=':', alpha=0.5)
ax.scatter(fluxonium_EJ_EC * 10, np.abs(fluxonium_anh)*100, s=80, c='r',
           marker='s', zorder=5, label='Fluxonium (punto óptimo)')
ax.set_xlabel('E_J / E_C'); ax.set_ylabel('|Anharmonicidad| (%)')
ax.set_title('Anharmonicidad: Transmon vs Fluxonium')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('Transmon EJ/EC=50:', f'{abs(anharmonicidad(50))*100:.1f}%')
print('Fluxonium típico:  ~40-70%')
print('Implicación: fluxonium permite puertas más selectivas, menos leakage')


## 2. Tabla comparativa de plataformas hardware 2024-2028


In [ ]:
import pandas as pd

plataformas = {
    'Plataforma': ['Transmon (IBM Heron)', 'Transmon (Google Willow)', 'Fluxonium (MIT/Yale)',
                   'Trapped Ion (IonQ Forte)', 'Spin-Si (Intel/Delft)', 'Majorana (Microsoft)'],
    'T1 (μs)': ['>300', '>100', '>1000', '>1e6', '~10-100', 'N/A (topológico)'],
    'T2 (μs)': ['>300', '~100', '>500', '>1e6', '~1-10', 'N/A'],
    'Error 1Q (%)': ['0.03', '0.05', '<0.01', '0.03', '0.1-1', '<0.001 (proyectado)'],
    'Error 2Q (%)': ['0.1', '0.3', '0.1-0.5', '0.3', '1-5', '<0.01 (proyectado)'],
    'Freq. puerta (ns)': ['20-50', '10-30', '50-200', '500-1000', '50-100', '>1000'],
    'Conectividad': ['Heavy-hex', 'All-to-all (Sycamore)', 'Variable', 'All-to-all', 'Línea', 'Topológica'],
    'Año operativo': ['2023', '2024', '2024-2025', '2023', '2025-2027', '2028-2030'],
}

df = pd.DataFrame(plataformas)
print(df.to_string(index=False))


## 3. Modelo de errores: Transmon vs Fluxonium

El error de puerta depende del tiempo de coherencia $T_1, T_2$ y del tiempo de puerta $t_g$:

$$\varepsilon_{1Q} \approx \frac{t_g}{T_1} + \frac{t_g}{T_2^*}$$

La alta anharmonicidad del fluxonium permite $t_g$ más cortos con menos leakage.


In [ ]:
def error_puerta_1q(T1_us: float, T2_us: float, tg_ns: float) -> float:
    """Error estimado de puerta de 1 qubit (modelo simplificado)."""
    tg_us = tg_ns / 1000
    return tg_us / T1_us + tg_us / T2_us

def error_puerta_2q(T1_us: float, T2_us: float, tg_ns: float,
                    anh_pct: float) -> float:
    """
    Error de puerta de 2 qubits, incluyendo contribución de leakage.
    leakage ~ (tg / anh)^2 donde anh en unidades de frecuencia.
    """
    tg_us = tg_ns / 1000
    err_dephasing = tg_us / T1_us + tg_us / T2_us
    err_leakage   = (tg_ns / (1000 * abs(anh_pct) / 100 * 100))**2  # heurístico
    return err_dephasing + err_leakage

# Parámetros típicos
qubits = {
    'Transmon IBM Heron': {'T1': 400, 'T2': 300, 'tg1': 35, 'tg2': 68, 'anh': 5},
    'Fluxonium MIT':      {'T1': 1000,'T2': 500, 'tg1': 80, 'tg2': 200,'anh': 50},
    'Spin-Si Delft':      {'T1': 50,  'T2': 5,   'tg1': 50, 'tg2': 100,'anh': 100},
}

print(f'{"Qubit":25s} | {"Err 1Q (%)":>10} | {"Err 2Q (%)":>10}')
print('-' * 52)
for nombre, p in qubits.items():
    e1 = error_puerta_1q(p['T1'], p['T2'], p['tg1']) * 100
    e2 = error_puerta_2q(p['T1'], p['T2'], p['tg2'], p['anh']) * 100
    print(f'{nombre:25s} | {e1:>10.4f} | {e2:>10.4f}')


## 4. Qubits Majorana: protección topológica

Los qubits topológicos codifican la información en **pares de fermiones de Majorana**
que no pueden ser perturbados localmente (el error requiere transportar una excitación
de un extremo del cable al otro).


In [ ]:
modelo_majorana = """
QUBITS DE MAJORANA: PRINCIPIOS CLAVE
═════════════════════════════════════════════════════════

Física base:
  • Un nanohilo semiconductorInAs/InSb en campo magnético + superconductor
    desarrolla modos de Majorana γ₁, γ₂ en los extremos del hilo.
  • El qubit se codifica en la paridad fermiónica: f = (γ₁ + iγ₂)/2
  • Las operaciones requieren braiding (intercambio de Majoranas)

Ventajas:
  • Error de decoherencia: O(e^{-L/ξ}) — exponencialmente suprimido con la longitud L
  • No necesita corrección de errores de ciclo activo para errores locales
  • 1 qubit lógico ≈ 1-2 qubits físicos (vs ~1000 en surface code)

Estado actual (2024-2025):
  • Microsoft Majorana 1: primer chip topológico operativo con 8 qubits
  • Error 2Q proyectado: < 0.01% (vs 0.1-0.5% transmon)
  • Frecuencia de puerta: > 1 μs (más lento que transmon)
  • Requisito: T < 100 mK, campo B ~100 mT

Roadmap:
  2025: demostración de braiding en 4 Majoranas
  2027: qubit lógico protegido topológico
  2030: procesador de 100 qubits topológicos

Desafíos actuales:
  • Gap topológico pequeño → necesita temperaturas muy bajas
  • Fabricación reproducible de heteroestructuras
  • Puertas de Clifford universales por braiding requieren 6+ Majoranas
  • Puerta T no se obtiene por braiding → necesita destilación de magic states
"""
print(modelo_majorana)


## 5. Simulación: ventaja del error intrínseco

Comparamos cuántos qubits físicos necesita cada plataforma para un qubit lógico
de calidad equivalente usando surface code.


In [ ]:
def qubits_fisicos_para_logico(p_fisico: float, p_logico_obj: float = 1e-6) -> int | None:
    """Qubits físicos necesarios para 1 qubit lógico con surface code."""
    p_th = 0.01
    if p_fisico >= p_th: return None
    if p_fisico <= 0: return 1
    ratio = p_fisico / p_th
    d = max(3, int(np.ceil(2 * np.log(p_logico_obj / 0.1) / np.log(ratio) - 1)))
    if d % 2 == 0: d += 1
    return 2 * d**2

plataformas_error = {
    'Transmon IBM':    0.001,   # 0.1%
    'Fluxonium':       0.0003,  # 0.03%
    'Spin-Si':         0.005,   # 0.5%
    'Majorana (proj)': 0.0001,  # 0.01%
    'Ion trap':        0.001,   # 0.1%
}

print(f'{"Plataforma":20s} | {"Error 2Q":>10} | {"Qubits/lógico":>15} | {"Factor vs transmon":>20}')
print('-' * 75)
q_transmon = qubits_fisicos_para_logico(plataformas_error['Transmon IBM'])
for nombre, p in plataformas_error.items():
    q = qubits_fisicos_para_logico(p)
    factor = q_transmon / q if q else 'N/A'
    print(f'{nombre:20s} | {p*100:>9.3f}% | {str(q):>15} | {str(round(factor,1) if isinstance(factor,float) else factor):>20}')

# Visualización
p_vals = np.logspace(-4, -2, 100)
q_vals = [qubits_fisicos_para_logico(p) for p in p_vals]
q_vals_clean = [q if q else None for q in q_vals]

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(p_vals*100, q_vals_clean, 'b-', lw=2)
for nombre, p in plataformas_error.items():
    q = qubits_fisicos_para_logico(p)
    if q: ax.scatter([p*100], [q], s=100, zorder=5, label=f'{nombre} ({q}q)')

ax.set_xlabel('Error físico 2Q (%)'); ax.set_ylabel('Qubits físicos por qubit lógico')
ax.set_title('Overhead de surface code vs calidad del hardware')
ax.legend(fontsize=8, loc='upper left'); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()


## Conclusiones

- El **fluxonium** supera al transmon en coherencia (T₁ > 1 ms) y anharmonicidad.
- Los **qubits Spin-Si** son más ruidosos actualmente pero ofrecen integración CMOS y alta densidad.
- Los **qubits Majorana** prometen protección topológica intrínseca, reduciendo el overhead de surface code 10-100×.
- El overhead de qubits físicos/lógico es **el factor limitante** hacia la computación cuántica útil.
